# Data Visualization — Quiz (Lectures 1–7)

**Time:** 90 minutes 
**Tools:** Plotly only (`plotly.express` first, then `update_traces` / `update_layout` / `update_xaxes` / `update_yaxes`)

**Your task:** answer each analytical question with **one polished, explanatory visualization**. For every question **you decide which chart type fits best** — and justify the choice in one or two sentences.

**Every figure must have:**
1. An **insight title** that states the finding, not just the variables
2. `template='plotly_white'` and a decluttered layout (no unnecessary gridlines, legends or axis titles)
3. A **CVD-safe palette** — no red/green; grey for context + one accent colour where it helps
4. Colour that **encodes meaning**, not decoration
5. Direct labels or annotations where they help the reader

**Chart types covered in the course** — choose from these:
line chart · sorted bar chart · stacked / 100% stacked bar chart · scatter plot · bubble chart · histogram · box plot · violin plot · choropleth map · treemap

> 💡 Each question has one chart type that fits it best, and the questions together cover most of the list above. If two of your answers use the same chart type, check the question again.


**Submit:** this notebook pushed to your personal repo as `quiz/dataviz_quiz.ipynb`

## The dataset — World Population 1970–2022

**Source:** [World Population Dataset](https://www.kaggle.com/datasets/iamsouravbanerjee/world-population-dataset) on Kaggle, compiled from Worldometer (UN World Population Prospects estimates). **234 rows × 16 columns — one row per country or territory, no missing values.**

| Column | Type | Description |
|---|---|---|
| `Country`, `Capital` | categorical | Country / territory and its capital |
| `Continent` | categorical | Africa, Asia, Europe, North America, Oceania, South America |
| `CCA3` | spatial | ISO-3 country code — use with `locationmode='ISO-3'` |
| `1970 Population` … `2022 Population` | temporal | Population in 1970, 1980, 1990, 2000, 2010, 2015, 2020, 2022 |
| `Area` | numerical | Land area in km² |
| `Density` | numerical | People per km² (2022) |
| `World share (%)` | numerical | Share of world population in 2022 |
| `Rank` | numerical | Rank by 2022 population |

**The next setup cell also gives you `df_long` — the same data in **tidy (long) format** with one row per country per year. Use `df` for one-year questions and `df_long` for questions about change over time**

In [1]:
import pandas as pd
import plotly.express as px
import numpy as np

df = pd.read_csv('world_population.csv')

# Shorter column names
df = df.rename(columns={'Country/Territory': 'Country', 'Area (km²)': 'Area',
                        'Density (per km²)': 'Density', 'World Population Percentage': 'World share (%)'})

# Tidy (long) version: one row per country per year
year_cols = [c for c in df.columns if c.endswith('Population')]
df_long = df.melt(id_vars=['CCA3', 'Country', 'Continent'], value_vars=year_cols,
                  var_name='Year', value_name='Population')

df_long['Year'] = df_long['Year'].str[:4].astype(int)     # '2022 Population' → 2022

print(df.shape, df_long.shape)


(234, 16) (1872, 5)


In [2]:
import plotly.graph_objects as go

ACCENT = '#E69F00'
GREY = '#C4C4C4'
INK = '#2B2B2B'
CONTINENT_COLORS = {
    'Africa': '#E69F00', 'Asia': '#0072B2', 'Europe': '#CC79A7',
    'North America': '#56B4E9', 'Oceania': '#8C8C8C', 'South America': '#6A51A3'}


def polish(fig, title, subtitle=None, height=540, showlegend=True):
    sub = f'<br><span style="font-size:13px;color:#6B6B6B">{subtitle}</span>' if subtitle else ''
    fig.update_layout(
        title=dict(text=f'<b>{title}</b>{sub}', x=0.01, xanchor='left',
                   font=dict(size=19, color=INK)),
        template='plotly_white', height=height,
        margin=dict(l=70, r=50, t=112 if subtitle else 84, b=55),
        font=dict(family='Segoe UI, Arial', color=INK, size=13),
        plot_bgcolor='white', paper_bgcolor='white', showlegend=showlegend,
        hoverlabel=dict(bgcolor='white', font_size=12))
    fig.update_xaxes(showgrid=False, zeroline=False, showline=True,
                     linecolor='#DCDCDC', ticks='outside', tickcolor='#DCDCDC')
    fig.update_yaxes(showgrid=True, gridcolor='#EFEFEF', zeroline=False, showline=False)
    return fig

---
## Q1 — Which continent is growing fastest?

How has the population of each continent **grown since 1970, relative to its 1970 size** — and which continent is growing fastest?


**Chart choice & justification:**I created a line chart to show changes over time. I set every continent's 1970 value to 100 to make growth easier to compare. I highlighted Africa in colour while keeping the other continents grey for comparison

In [3]:
tot = df_long.groupby(['Year', 'Continent'], as_index=False)['Population'].sum()
piv = tot.pivot(index='Year', columns='Continent', values='Population')
indexed = piv / piv.loc[1970] * 100
right = indexed.iloc[-1].sort_values()
colors = {c: (ACCENT if c == 'Africa' else GREY) for c in indexed.columns}
fig1 = go.Figure()
for c in right.index:
    fig1.add_trace(go.Scatter(
        x=indexed.index, y=indexed[c], mode='lines', name=c,
        line=dict(color=colors[c], width=4 if c == 'Africa' else 1.8),
        hovertemplate=f'{c}<br>%{{x}}: %{{y:.0f}} (1970 = 100)<extra></extra>'))
    if c in ['Africa', 'Asia', 'Europe']:
        fig1.add_annotation(x=2022, y=right[c], text=f'  {c}', showarrow=False,
                            xanchor='left', yshift=-8 if c == 'Asia' else 0,
                            font=dict(size=12, color='#555555'))
fig1.add_hline(y=100, line=dict(color='#B0B0B0', width=1, dash='dot'))
fig1.add_annotation(x=1974, y=104, text='1970 level', showarrow=False,
                    font=dict(size=11, color='#8A8A8A'))
polish(fig1, 'Africa\u2019s population has nearly quadrupled since 1970 \u2014 Europe\u2019s has barely moved',
       'Population in each year as an index of its 1970 level (1970 = 100) \u00b7 grey = context, orange = Africa', showlegend=False)
fig1.update_xaxes(range=[1968, 2033], tickmode='array',
                  tickvals=[1970, 1980, 1990, 2000, 2010, 2022])
fig1.update_yaxes(title='Population (1970 = 100)')

---
## Q2 — Which countries added the most people?

Which **15 countries added the most people** between 1970 and 2022 — and how many of them are in **Africa**?

**Chart choice & justification:** I created a sorted horizontal bar chart to compare population growth across countries. I highlighted African countries in orange and kept the others grey with values shown at the end of each bar for easy comparison

In [4]:
top = (df.assign(Added=(df['2022 Population'] - df['1970 Population']) / 1e6)
         .nlargest(15, 'Added').sort_values('Added'))
bar_colors = [ACCENT if c == 'Africa' else GREY for c in top['Continent']]
fig2 = go.Figure(go.Bar(
    x=top['Added'], y=top['Country'], orientation='h',
    marker=dict(color=bar_colors, line=dict(width=0)),
    text=[f'{v:,.0f}m' for v in top['Added']], textposition='outside',
    textfont=dict(size=11, color='#555555'),
    customdata=top['Continent'],
    hovertemplate='%{y}<br>Added: %{x:,.0f} million<br>Continent: %{customdata}<extra></extra>'))
polish(fig2, 'India and China added 1.5 billion people \u2014 4 of the top 15 are African',
       'Population added 1970\u20132022, top 15 countries (millions) \u00b7 orange = Africa', height=620, showlegend=False)
fig2.update_xaxes(title='People added 1970\u20132022 (millions)', range=[0, 960])
fig2.update_yaxes(title=None, showgrid=False)

---
## Q3 — Do bigger countries have more people?

Is there a relationship between a country's **land area** and its **2022 population** — does it differ by **continent**, and which countries clearly break the pattern?

**Chart choice & justification:** I created a scatter plot to show the relationship between a country's area and population and to spot unusual countries. I used log scales because the values vary a lot and different colours to show each continent, with labels for the most unusual countries

In [5]:
focus = ['Russia', 'Canada', 'China', 'United States', 'Brazil',
         'Macau', 'Monaco', 'Singapore', 'Hong Kong']
scatter = df.copy()
scatter['label'] = np.where(scatter['Country'].isin(focus), scatter['Country'], '')
fig3 = px.scatter(scatter, x='Area', y='2022 Population', color='Continent',
                  color_discrete_map=CONTINENT_COLORS, log_x=True, log_y=True,
                  hover_name='Country', custom_data=['Density'],
                  labels={'Area': 'Land area (km\u00b2, log scale)',
                          '2022 Population': 'Population in 2022 (log scale)',
                          'Continent': 'Continent'})
fig3.update_traces(marker=dict(size=9, line=dict(width=0.6, color='white')),
                   text=scatter['label'], textposition='top center',
                   textfont=dict(size=10, color='#444444'))
fig3.update_traces(hovertemplate='%{hovertext}<br>Area: %{x:,.0f} km\u00b2'
                  '<br>Population: %{y:,.0f}<br>Density: %{customdata[0]:,.0f}/km\u00b2<extra></extra>')
polish(fig3, 'Land area barely predicts population \u2014 Asia\u2019s tiny city-states are extreme outliers',
       'Each dot is a country or territory \u00b7 log scales \u00b7 labelled points are the clearest outliers', showlegend=True)
fig3.add_annotation(x=30, y=695168, text='Macau: 23,000/km\u00b2',
                    showarrow=True, arrowhead=2, arrowcolor='#8A8A8A', ax=70, ay=-35,
                    font=dict(size=11, color='#555555'))
fig3.update_xaxes(title='Land area (km\u00b2, log scale)', range=[-0.2, 7.6])
fig3.update_yaxes(title='Population in 2022 (log scale)', range=[2.5, 9.7])

---
## Q4 — How crowded are the continents?

How does **population density vary within and across continents**? Show the spread, not just the average.

**Chart choice & justification:** I created a box plot to compare population density across continents and show the median and spread. I used a log scale because the values vary a lot and highlighted Asia in orange because it has the widest range while labelling unusual values for clarity

In [6]:
order = df.groupby('Continent')['Density'].median().sort_values().index.tolist()
box_colors = {c: (ACCENT if c == 'Asia' else GREY) for c in order}
fig4 = px.box(df, x='Continent', y='Density', color='Continent',
              color_discrete_map=box_colors, category_orders={'Continent': order},
              log_y=True, points='outliers', hover_name='Country',
              labels={'Density': 'Population density (people per km\u00b2, log scale)'})
fig4.update_traces(width=0.55, marker=dict(size=5, opacity=0.8),
                   line=dict(width=1.5))
polish(fig4, 'Asia\u2019s density spread dwarfs every other continent \u2014 from 2 to 23,000 people per km\u00b2',
       'Box = middle 50% of territories, line = median, dots = outliers \u00b7 orange = Asia', height=560, showlegend=False)
fig4.add_annotation(x='Asia', y=23172, text='Macau 23,172', showarrow=True,
                    arrowhead=2, arrowcolor='#8A8A8A', ax=55, ay=-30,
                    font=dict(size=11, color='#555555'))
fig4.update_yaxes(title='Population density (people per km\u00b2, log scale)')
fig4.update_xaxes(title=None)

---
## Q5 — Where did populations grow — and where did they shrink?

Which countries' populations **grew and which shrank between 2000 and 2022** (percentage change), and where in the world are they?

**Chart choice & justification:** I created a choropleth map to show population changes around the world. Blue represents countries where the population is decreasing while orange shows countries where it is growing making the differences easy to see

In [7]:
chg = df.assign(Change=(df['2022 Population'] / df['2000 Population'] - 1) * 100)
scale = [[0.0, '#2166AC'], [0.25, '#E9E9E9'], [0.50, '#F3D9A4'], [1.0, '#E69F00']]
fig5 = px.choropleth(chg, locations='CCA3', locationmode='ISO-3', color='Change',
                     hover_name='Country', custom_data=['Continent', 'Change'],
                     color_continuous_scale=scale, range_color=(-40, 120),
                     labels={'Change': '% change'})
fig5.update_traces(hovertemplate='%{hovertext}<br>%{customdata[0]}'
                   '<br>Change 2000\u20132022: %{customdata[1]:+.1f}%<extra></extra>')
fig5.update_geos(projection_type='natural earth', showframe=False, showcoastlines=False,
                 showcountries=True, countrycolor='white', landcolor='#F1F1F1',
                 bgcolor='white', lataxis_range=[-58, 85])
fig5.update_layout(coloraxis_colorbar=dict(
    title=dict(text='% change<br>2000\u201322', font=dict(size=11)),
    orientation='h', y=-0.09, x=0.5, xanchor='center', thickness=10, len=0.5,
    tickfont=dict(size=10), outlinewidth=0))
polish(fig5, 'Since 2000, the Gulf and West Africa surged while Eastern Europe and the Pacific shrank',
       'Population change 2000\u20132022 by country (%) \u00b7 blue = shrinking, orange = growing \u00b7 34 countries shrank',
       height=600, showlegend=False)
fig5.update_layout(margin=dict(l=10, r=10, t=112, b=65))

---
## Q6 — Where do the world's 8 billion people live?

How is the **2022 world population** distributed across **continents and the countries within them**? Show both levels in one figure.

**Chart choice & justification:** I created a treemap to show how the world's population is divided between continents and countries. Bigger rectangles represent larger populations and different colours make each continent easy to identify

In [8]:
tree = df[df['2022 Population'] > 0].copy()
fig6 = px.treemap(tree, path=[px.Constant('World'), 'Continent', 'Country'],
                  values='2022 Population', color='Continent',
                  color_discrete_map=CONTINENT_COLORS,
                  custom_data=['Country'])
fig6.update_traces(texttemplate='<b>%{label}</b><br>%{percentRoot:.1%} of world',
                   hovertemplate='<b>%{label}</b><br>Population 2022: %{value:,.0f}'
                   '<br>Share of world: %{percentRoot:.1%}<extra></extra>',
                   root_color='#F2F2F2', tiling=dict(pad=2))
fig6.update_layout(coloraxis_showscale=False)
polish(fig6, 'Asia holds 59% of the world\u2019s people \u2014 China and India alone hold more than a third',
       '2022 population by continent and country \u00b7 area = number of people \u00b7 labels show share of world total',
       height=660, showlegend=False)
fig6.update_layout(margin=dict(l=10, r=10, t=112, b=10))

---
## Q7 — Whose share of the world is rising?

How have the shares of world population held by **Africa, Asia and Europe** changed from 1970 to 2022? Group the other three continents as `'Rest of the world'`.

**Chart choice & justification:**I created a 100% stacked bar chart to show how each continent's share of the world's population changes over time. Each year adds up to 100% and different colours make the continents easy to compare.

In [9]:
piv7 = df_long.groupby(['Year', 'Continent'], as_index=False)['Population'].sum()
piv7['Group'] = piv7['Continent'].where(piv7['Continent'].isin(['Africa', 'Asia', 'Europe']),
                                        'Rest of the world')
shares = piv7.groupby(['Year', 'Group'], as_index=False)['Population'].sum()
shares['Share'] = shares['Population'] / shares.groupby('Year')['Population'].transform('sum') * 100
group_order = ['Africa', 'Asia', 'Europe', 'Rest of the world']
group_colors = {'Africa': ACCENT, 'Asia': '#0072B2', 'Europe': '#CC79A7',
                'Rest of the world': '#D0D0D0'}
shares['YearLabel'] = shares['Year'].astype(str)
fig7 = px.bar(shares, x='YearLabel', y='Share', color='Group', barmode='stack',
              color_discrete_map=group_colors, category_orders={'Group': group_order},
              labels={'Share': 'Share of world population (%)', 'YearLabel': 'Year'},
              custom_data=['Group'])
fig7.update_traces(marker=dict(line=dict(width=0.5, color='white')),
                   hovertemplate='%{x} \u00b7 %{customdata[0]}: %{y:.1f}%<extra></extra>')
polish(fig7, 'Africa\u2019s share of world population has nearly doubled since 1970 \u2014 Europe\u2019s has nearly halved',
       'Share of world population (%) \u00b7 Africa 9.9% \u2192 17.9%; Europe 17.8% \u2192 9.3%', height=560, showlegend=True)
fig7.update_yaxes(title='Share of world population (%)', range=[0, 100], ticksuffix='%')
fig7.update_xaxes(title=None)
fig7.update_layout(legend=dict(orientation='h', y=-0.12, x=0, title=None,
                               font=dict(size=12)))